# 🤖 Agente IA NexusEcom — RAG Corporativo
> **Challenge Alura Agente | E-Commerce Inteligente**

🏢 **Empresa:** NexusEcom — *Tienda Online Multiplataforma*  
🛠️ **Stack Tecnologico:** Cohere `command-r-plus-08-2024` • ChromaDB • LangChain • Streamlit • OCI  

---

### 📑 Roadmap del Proyecto

| Fase | Descripcion | Estado |
| :---: | :--- | :---: |
| **0** | ⚙️ Setup y configuracion del entorno | ✅ Completado |
| **1** | 📁 Colecta y organizacion de documentos | ✅ Completado |
| **2** | 🧠 Procesamiento y extraccion de contenido | 🔄 En proceso |
| **3** | 🗄️ Indexacion vectorial | ⏳ Pendiente |
| **4** | 🔍 Capa RAG (Recuperacion) | ⏳ Pendiente |
| **5** | 💬 Generacion y validacion de respuestas | ⏳ Pendiente |
| **6** | 🖥️ Interfaz Streamlit | ⏳ Pendiente |
| **7** | ☁️ Deploy en OCI | ⏳ Pendiente |
| **8** | 📊 Registro y cierre | ⏳ Pendiente |

---
## ⚙️ FASE 0 — Inicializacion y Verificacion

> ⚠️ **IMPORTANTE:** Ejecuta la **celda 0.1 PRIMERO** — define todas las variables globales.

In [18]:
# ============================================================
# Celda 0.1 — INICIALIZACION GLOBAL (ejecutar SIEMPRE primero)
# ============================================================
import os
import sys
import importlib
from pathlib import Path
from dotenv import load_dotenv

BASE_DIR = Path().resolve()
env_path = BASE_DIR / '.env'

if env_path.exists():
    load_dotenv(env_path, override=True)
    print(f'✅ .env cargado desde: {env_path}')
else:
    print(f'❌ Archivo .env no encontrado en: {env_path}')

COHERE_API_KEY     = os.getenv('COHERE_API_KEY', '')
LLM_MODEL          = os.getenv('LLM_MODEL', 'command-r-plus-08-2024')
EMBEDDING_MODEL    = os.getenv('EMBEDDING_MODEL', 'embed-multilingual-v3.0')
CHUNK_SIZE         = int(os.getenv('CHUNK_SIZE', 1000))
CHUNK_OVERLAP      = int(os.getenv('CHUNK_OVERLAP', 200))
MAX_RETRIEVAL_DOCS = int(os.getenv('MAX_RETRIEVAL_DOCS', 5))
CHROMA_DB_PATH     = os.getenv('CHROMA_DB_PATH', str(BASE_DIR / 'chroma_db'))
COLLECTION_NAME    = os.getenv('CHROMA_COLLECTION_NAME', 'nexusecom_documents')
DOCS_DIR           = BASE_DIR / 'docs'

if COHERE_API_KEY and COHERE_API_KEY not in ('tu_api_key_aqui', ''):
    print(f'✅ COHERE_API_KEY cargada (...{COHERE_API_KEY[-4:]})')
else:
    print('❌ COHERE_API_KEY no configurada — edita el archivo .env')

print('\n⚙️  Configuracion Activa:')
print(f'  🔹 LLM:              {LLM_MODEL}')
print(f'  🔹 Embeddings:       {EMBEDDING_MODEL}')
print(f'  🔹 Chunk size:       {CHUNK_SIZE}')
print(f'  🔹 Coleccion Chroma: {COLLECTION_NAME}')

✅ .env cargado desde: C:\Users\Danny\Documents\Challenge Alura Agente\.env
✅ COHERE_API_KEY cargada (...m4Jd)

⚙️  Configuracion Activa:
  🔹 LLM:              command-r-plus-08-2024
  🔹 Embeddings:       embed-multilingual-v3.0
  🔹 Chunk size:       1000
  🔹 Coleccion Chroma: nexusecom_documents


In [19]:
# ============================================================
# Celda 0.2 — Verificar librerias instaladas
# ============================================================
librerias = {
    'cohere': 'Cohere SDK',
    'langchain_cohere': 'LangChain-Cohere',
    'langchain': 'LangChain',
    'langchain_text_splitters': 'LangChain Text Splitters',
    'langchain_core': 'LangChain Core',
    'langchain_chroma': 'LangChain ChromaDB',
    'chromadb': 'ChromaDB',
    'fitz': 'PyMuPDF (PDF)',
    'docx': 'python-docx (Word)',
    'openpyxl': 'openpyxl (Excel)',
    'pptx': 'python-pptx (PowerPoint)',
    'streamlit': 'Streamlit',
    'pandas': 'Pandas',
    'loguru': 'Loguru',
}
print(f'🐍 Python: {sys.version}')
print('=' * 50)
errores = []
for modulo, nombre in librerias.items():
    try:
        importlib.import_module(modulo)
        print(f'✅ {nombre:30}')
    except ImportError:
        print(f'❌ {nombre:30}')
        errores.append(nombre)
print('=' * 50)
print('🚀 TODAS LAS LIBRERIAS OK' if not errores else f'⚠️ ERRORES EN: {errores}')

🐍 Python: 3.13.12 | packaged by Anaconda, Inc. | (main, Feb 24 2026, 16:05:56) [MSC v.1942 64 bit (AMD64)]
✅ Cohere SDK                    
✅ LangChain-Cohere              
✅ LangChain                     
✅ LangChain Text Splitters      
✅ LangChain Core                
✅ LangChain ChromaDB            
✅ ChromaDB                      
✅ PyMuPDF (PDF)                 
✅ python-docx (Word)            
✅ openpyxl (Excel)              
✅ python-pptx (PowerPoint)      
✅ Streamlit                     
✅ Pandas                        
✅ Loguru                        
🚀 TODAS LAS LIBRERIAS OK


In [20]:
# ============================================================
# Celda 0.3 — Verificar conexion con Cohere LLM
# ============================================================
import cohere

try:
    co = cohere.ClientV2(api_key=COHERE_API_KEY)
    response = co.chat(
        model=LLM_MODEL,
        messages=[{'role': 'user', 'content': 'Responde solo: Conexion exitosa con Cohere'}]
    )
    print('✅ Conexion con Cohere verificada exitosamente')
    print(f'🧠 Modelo: {LLM_MODEL}')
    print(f'💬 Respuesta: {response.message.content[0].text.strip()}')
except Exception as e:
    print(f'❌ ERROR de conexion: {e}')

✅ Conexion con Cohere verificada exitosamente
🧠 Modelo: command-r-plus-08-2024
💬 Respuesta: Conexión exitosa con Cohere. ¿En qué puedo ayudarte?


In [21]:
# ============================================================
# Celda 0.4 — Verificar modelo de embeddings Cohere
# ============================================================
try:
    test_resp = co.embed(
        texts=['Politica de reembolsos NexusEcom'],
        model=EMBEDDING_MODEL,
        input_type='search_document',
        embedding_types=['float']
    )
    dim = len(test_resp.embeddings.float[0])
    print('✅ Modelo de Embeddings verificado')
    print(f'🧬 Modelo: {EMBEDDING_MODEL}')
    print(f'📐 Dimensiones: {dim}')
except Exception as e:
    print(f'❌ ERROR de Embeddings: {e}')

✅ Modelo de Embeddings verificado
🧬 Modelo: embed-multilingual-v3.0
📐 Dimensiones: 1024


---
## 📁 FASE 1 — Colecta y Organizacion de Documentos

In [22]:
# ============================================================
# Celda 1.1 — Inventario de documentos NexusEcom
# ============================================================
import pandas as pd

EXTENSIONES = {'.pdf', '.docx', '.xlsx', '.pptx', '.md', '.csv', '.json', '.html', '.txt'}

inventario = []
for archivo in DOCS_DIR.rglob('*'):
    if archivo.is_file() and archivo.suffix.lower() in EXTENSIONES:
        inventario.append({
            'archivo': archivo.name,
            'categoria': archivo.parent.name,
            'extension': archivo.suffix.lower(),
            'tamano_kb': round(archivo.stat().st_size / 1024, 2),
            'ruta': str(archivo.relative_to(BASE_DIR))
        })

if inventario:
    df_docs = pd.DataFrame(inventario)
    print(f'📂 Total documentos encontrados: {len(df_docs)}\n')
    print('📊 Distribucion por Categoria:')
    print(df_docs.groupby('categoria')['archivo'].count().to_string())
    print('\n📋 Detalles del Inventario:')
    display(df_docs)
else:
    print('⚠️ No se encontraron documentos en el directorio /docs')

📂 Total documentos encontrados: 5

📊 Distribucion por Categoria:
categoria
garantias_y_soporte    1
logistica              1
marketing_afiliados    1
pagos_y_facturacion    1
politicas              1

📋 Detalles del Inventario:


,archivo,categoria,extension,tamano_kb,ruta
0,Manual_Garantia_Productos_NexusEcom.pdf,garantias_y_soporte,.pdf,16.43,docs\garantias_y_soporte\Manual_Garantia_Produ...
1,Guia_Tiempos_Costos_Envio_NexusEcom.pdf,logistica,.pdf,19.40,docs\logistica\Guia_Tiempos_Costos_Envio_Nexus...
2,Programa_Afiliados_NexusEcom.pdf,marketing_afiliados,.pdf,17.10,docs\marketing_afiliados\Programa_Afiliados_Ne...
3,Preguntas_Frecuentes_Metodos_Pago_NexusEcom.pdf,pagos_y_facturacion,.pdf,17.83,docs\pagos_y_facturacion\Preguntas_Frecuentes_...
4,Politica_Reembolsos_NexusEcom.pdf,politicas,.pdf,24.53,docs\politicas\Politica_Reembolsos_NexusEcom.pdf


---
## 🧠 FASE 2 — Procesamiento y Extraccion de Contenido

In [23]:
# ============================================================
# Celda 2.1 — Extractor de PDF con PyMuPDF
# ============================================================
import fitz  # PyMuPDF

def extraer_texto_pdf(ruta_pdf: Path) -> dict:
    """Extrae texto de un PDF pagina por pagina con manejo de errores."""
    try:
        doc = fitz.open(str(ruta_pdf))
        paginas = []
        for num, pagina in enumerate(doc, start=1):
            texto = pagina.get_text('text').strip()
            if texto:
                paginas.append(f'[Pagina {num}]\n{texto}')
        doc.close()
        return {
            'archivo': ruta_pdf.name,
            'categoria': ruta_pdf.parent.name,
            'texto': '\n\n'.join(paginas),
            'num_paginas': len(paginas),
            'ruta': str(ruta_pdf),
            'extension': '.pdf'
        }
    except Exception as e:
        print(f'❌ ERROR en {ruta_pdf.name}: {e}')
        return None

documentos_procesados = []
pdfs = list(DOCS_DIR.rglob('*.pdf'))
print(f'📑 Analizando {len(pdfs)} PDFs...\n')

for pdf_path in pdfs:
    resultado = extraer_texto_pdf(pdf_path)
    if resultado:
        documentos_procesados.append(resultado)
        print(f'✅ {resultado["archivo"]:45s} | 📄 {resultado["num_paginas"]:2d} pag. | 🔤 {len(resultado["texto"])} chars')

print(f'\n🚀 Total procesados exitosamente: {len(documentos_procesados)}')

📑 Analizando 5 PDFs...

✅ Manual_Garantia_Productos_NexusEcom.pdf       | 📄  9 pag. | 🔤 11538 chars
✅ Guia_Tiempos_Costos_Envio_NexusEcom.pdf       | 📄 10 pag. | 🔤 14455 chars
✅ Programa_Afiliados_NexusEcom.pdf              | 📄  9 pag. | 🔤 12282 chars
✅ Preguntas_Frecuentes_Metodos_Pago_NexusEcom.pdf | 📄  9 pag. | 🔤 13420 chars
✅ Politica_Reembolsos_NexusEcom.pdf             | 📄 12 pag. | 🔤 20112 chars

🚀 Total procesados exitosamente: 5


In [24]:
# ============================================================
# Celda 2.2 — Chunking con LangChain
# ============================================================
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document

splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    separators=['\n\n', '\n', '. ', ' ', ''],
    length_function=len
)

chunks_totales = []
print('✂️ Dividiendo documentos en chunks semanticos...\n')

for doc in documentos_procesados:
    lc_doc = Document(
        page_content=doc['texto'],
        metadata={
            'archivo': doc['archivo'],
            'categoria': doc['categoria'],
            'extension': doc['extension'],
            'ruta': doc['ruta'],
            'empresa': 'NexusEcom'
        }
    )
    chunks = splitter.split_documents([lc_doc])
    chunks_totales.extend(chunks)
    print(f'🔹 {doc["archivo"]:45s} ➜ {len(chunks)} chunks')

print(f'\n🧩 Total de chunks generados: {len(chunks_totales)}')

if chunks_totales:
    print('\n🔍 Muestra del primer chunk:')
    print('━' * 60)
    print(chunks_totales[0].page_content[:250] + '...')
    print('━' * 60)
    print(f'🏷️ Metadata: {chunks_totales[0].metadata}')

✂️ Dividiendo documentos en chunks semanticos...

🔹 Manual_Garantia_Productos_NexusEcom.pdf       ➜ 17 chunks
🔹 Guia_Tiempos_Costos_Envio_NexusEcom.pdf       ➜ 20 chunks
🔹 Programa_Afiliados_NexusEcom.pdf              ➜ 17 chunks
🔹 Preguntas_Frecuentes_Metodos_Pago_NexusEcom.pdf ➜ 18 chunks
🔹 Politica_Reembolsos_NexusEcom.pdf             ➜ 27 chunks

🧩 Total de chunks generados: 99

🔍 Muestra del primer chunk:
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
[Pagina 1]
Manual de Garantía de Productos de NexusEcom
Índice
1. Propósito
2. Alcance
3. Objetivo operativo
4. Definiciones
5. Plazos de garantía
6. Cobertura general
7. Exclusiones
8. Tipos de resolución
9. Procedimiento de evaluación
10. Evidencia...
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
🏷️ Metadata: {'archivo': 'Manual_Garantia_Productos_NexusEcom.pdf', 'categoria': 'garantias_y_soporte', 'extension': '.pdf', 'ruta': 'C:\\Users\\Danny\\Documents\\Challenge Alura Agente\\docs\\garantias_y_sopor

---
## 🗄️ FASE 3 — Indexacion Vectorial con Cohere + ChromaDB

In [25]:
# ============================================================
# Celda 3.1 — Indexar chunks en ChromaDB con embeddings Cohere
# ============================================================
from langchain_cohere import CohereEmbeddings
from langchain_chroma import Chroma

embeddings = CohereEmbeddings(
    cohere_api_key=COHERE_API_KEY,
    model=EMBEDDING_MODEL
)
print(f'🧬 Configurando Embeddings: {EMBEDDING_MODEL}')
print(f'📥 Indexando {len(chunks_totales)} chunks en ChromaDB...')

try:
    vectorstore = Chroma.from_documents(
        documents=chunks_totales,
        embedding=embeddings,
        collection_name=COLLECTION_NAME,
        persist_directory=CHROMA_DB_PATH
    )
    total = vectorstore._collection.count()
    print(f'✅ Indexacion exitosa ➜ {total} vectores persistidos en DB')
except Exception as e:
    print(f'❌ ERROR durante la indexacion: {e}')
    raise

🧬 Configurando Embeddings: embed-multilingual-v3.0
📥 Indexando 99 chunks en ChromaDB...
✅ Indexacion exitosa ➜ 297 vectores persistidos en DB


In [26]:
# ============================================================
# Celda 3.2 — Prueba de Busqueda Semantica
# ============================================================
preguntas_prueba = [
    'Como solicitar un reembolso?',
    'Cuanto tiempo tarda el envio?',
    'Como unirse al programa de afiliados?',
    'Que metodos de pago aceptan?',
    'Como aplica la garantia de un producto?'
]

print('🎯 TEST DE BUSQUEDA SEMANTICA\n')
for pregunta in preguntas_prueba:
    resultados = vectorstore.similarity_search(pregunta, k=1)
    if resultados:
        doc = resultados[0]
        print(f'🗣️ Pregunta:  {pregunta}')
        print(f'📄 Documento: {doc.metadata["archivo"]}')
        print(f'📝 Fragmento: {doc.page_content[:120]}...')
        print('━' * 60)

🎯 TEST DE BUSQUEDA SEMANTICA

🗣️ Pregunta:  Como solicitar un reembolso?
📄 Documento: Politica_Reembolsos_NexusEcom.pdf
📝 Fragmento: El cliente puede solicitar devolución por retracto dentro de los 10 días corridos posteriores a la recepción
del pedido,...
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
🗣️ Pregunta:  Cuanto tiempo tarda el envio?
📄 Documento: Guia_Tiempos_Costos_Envio_NexusEcom.pdf
📝 Fragmento: [Pagina 3]
•
Tamaño y peso del producto
•
Inventario local o regional
•
Validaciones de pago
•
Eventos operativos o clim...
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
🗣️ Pregunta:  Como unirse al programa de afiliados?
📄 Documento: Programa_Afiliados_NexusEcom.pdf
📝 Fragmento: [Pagina 2]
Incluye:
•
Promoción digital
•
Contenidos editoriales
•
Campañas con código o enlace
•
Seguimiento de atribuc...
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
🗣️ Pregunta:  Que metodos de pago aceptan?
📄 Documento: Preguntas_Frecuentes_Metodos_Pago_N

---
## 🔍 FASE 4 — Capa de Recuperacion RAG

In [27]:
# ============================================================
# Celda 4.1 — Configurar Retriever
# ============================================================
retriever = vectorstore.as_retriever(
    search_type='similarity',
    search_kwargs={'k': MAX_RETRIEVAL_DOCS}
)
print(f'✅ Retriever Activado ➜ Recuperando los Top-{MAX_RETRIEVAL_DOCS} documentos por consulta')

✅ Retriever Activado ➜ Recuperando los Top-5 documentos por consulta


---
## 💬 FASE 5 — Generacion de Respuestas con Cohere RAG

In [28]:
# ============================================================
# Celda 5.1 — Cadena RAG con Cohere + LangChain
# ============================================================
from langchain_cohere import ChatCohere
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

llm = ChatCohere(
    cohere_api_key=COHERE_API_KEY,
    model=LLM_MODEL,
    temperature=0.1
)

SYSTEM_PROMPT = """Eres el asistente virtual oficial de NexusEcom, una tienda online de e-commerce.
Tu funcion es responder preguntas basandote UNICAMENTE en los documentos internos proporcionados.

Reglas Corporativas:
1. Responde SOLO con informacion del contexto proporcionado.
2. Indica siempre el documento fuente.
3. Si no encuentras la informacion, responde: "No encontre esta informacion en los documentos disponibles."
4. Responde en espanol, de forma clara, amable y corporativa.
5. No inventes datos, fechas, precios ni politicas.

Contexto Recuperado:
{context}
"""

prompt = ChatPromptTemplate.from_messages([
    ('system', SYSTEM_PROMPT),
    ('human', '{question}')
])

def formatear_contexto(docs):
    partes = []
    for i, doc in enumerate(docs, 1):
        fuente = doc.metadata.get('archivo', 'Desconocido')
        categoria = doc.metadata.get('categoria', '')
        partes.append(f'[Fuente {i}: {fuente} | {categoria}]\n{doc.page_content}')
    return '\n\n---\n\n'.join(partes)

cadena_rag = (
    {'context': retriever | formatear_contexto, 'question': RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

print(f'✅ Cadena RAG Ensamblada | 🧠 Modelo: {LLM_MODEL}')

✅ Cadena RAG Ensamblada | 🧠 Modelo: command-r-plus-08-2024


In [29]:
# ============================================================
# Celda 5.2 — Prueba Final del Agente NexusEcom
# ============================================================
preguntas_test = [
    'Como puedo solicitar la devolucion de un producto?',
    'Cuales son los tiempos de entrega?',
    'Que comision ofrece el programa de afiliados?',
    'Cuales son los metodos de pago disponibles?',
    'Como funciona la garantia de productos electronicos?'
]

print('🤖 === TEST FINAL AGENTE NEXUSECOM === 🤖')
for i, pregunta in enumerate(preguntas_test, 1):
    print(f'\n🔹 --- Pregunta {i} ---')
    print(f'👤 Usuario: {pregunta}')
    try:
        respuesta = cadena_rag.invoke(pregunta)
        print(f'🤖 Agente:  {respuesta}')
    except Exception as e:
        print(f'❌ ERROR: {e}')
    print('━' * 60)

🤖 === TEST FINAL AGENTE NEXUSECOM === 🤖

🔹 --- Pregunta 1 ---
👤 Usuario: Como puedo solicitar la devolucion de un producto?
🤖 Agente:  Según la política de reembolsos de NexusEcom, los clientes pueden solicitar la devolución de un producto en diferentes situaciones:

1. **Retracto de compra**: El cliente tiene un plazo de 10 días corridos posteriores a la recepción del pedido para solicitar la devolución, siempre que el producto cumpla con los requisitos de elegibilidad.

2. **Producto incorrecto, faltante o con daño visible**: En estos casos, la solicitud debe realizarse dentro de las 48 horas posteriores a la entrega, proporcionando evidencia fotográfica o video que demuestre la incidencia.

3. **Falla de funcionamiento**: La devolución por falla de funcionamiento debe solicitarse dentro del período de garantía aplicable al producto, según lo establecido en el Manual de Garantía de Productos de NexusEcom.

Para iniciar el proceso de devolución, es importante que te comuniques con el 

---
## 🖥️ FASE 6 — Interfaz Streamlit
> 📝 Desarrollada en `src/interface/app.py`  
> 🚀 Ejecutar con: `streamlit run src/interface/app.py`

---
## ☁️ FASE 7 — Deploy en OCI
> 🌐 Documentacion de despliegue en Oracle Cloud Infrastructure

---
## 📊 FASE 8 — Registro de Ejecucion y Cierre
> 📸 Evidencia visual del agente funcionando en produccion OCI